# HORIZON - training notebook

Trains the world model and writes the artifacts the backend (`horizon-api/`) loads:

| file | required | consumed by |
| --- | --- | --- |
| `states.parquet` | yes | history slicing, host list, surprise |
| `scaler.json` | yes | feature transforms |
| `model.pt` | yes | the model |
| `metrics.json` | no | `GET /api/metrics` |
| `scenarios.json` | no | demo-host ground truth |
| `model_heldout_<class>.pt` | no | held-out-class surprise overlay |

Everything is written to **`artifacts/`** in the working dir. Download that folder and
drop it into `horizon-api/artifacts/` (see the last cell).

---

## Run in Kaggle

1. New Notebook -> **Add Input** -> search `chethuhn/network-intrusion-dataset` -> Add.
2. Settings -> Accelerator: **GPU T4** (optional, CPU works for a short run).
3. Run all. Artifacts appear in `/kaggle/working/artifacts/`; download from the Output tab.

## Run in Colab

1. Runtime -> Change runtime type -> **T4 GPU** (optional).
2. Run the first cells. When prompted, upload your `kaggle.json`
   (kaggle.com -> Account -> Create New API Token) so the dataset can download.
3. Artifacts appear in `/content/artifacts/`; the last cell zips them for download.

## Prerequisite

`horizon-api/` must be pushed to the GitHub repo below - the notebook clones it so
training and serving share one model definition. If you have not pushed it yet, do
that first.

In [ ]:
# ============================ CONFIG ============================
REPO_URL            = "https://github.com/haragam22/HORIZON.git"

HELDOUT_CAPTURE     = "ids2017-friday"   # domain-shift day; excluded from training, used for eval
WINDOW_SECONDS      = 60
MIN_WINDOWS         = 40                 # drop hosts with fewer real windows
WARMUP_WINDOWS      = 5                  # skip first N windows/host when fitting the scaler (new_peer_rate bias)
LABEL_MIN_MALICIOUS = 1                  # malicious flows in a window before it counts as an attack window

EPOCHS              = 15                 # 40 for a real run
BATCH_SIZE          = 256
LR                  = 1e-3
LAMBDA_BCE          = 1.0
N_ROLLOUT_SAMPLES   = 50

QUICK               = False             # True -> sample 15% of flows for a fast smoke run
RUN_HELDOUT_CLASS   = False             # True -> also retrain with one attack class removed (slow: +1 full train)
HELDOUT_CLASS       = "PortScan"
SEED                = 0

In [ ]:
import os, sys, pathlib, json, time
import numpy as np, pandas as pd
import torch

torch.manual_seed(SEED); np.random.seed(SEED)

IN_KAGGLE = pathlib.Path("/kaggle").exists()
IN_COLAB  = "google.colab" in sys.modules or pathlib.Path("/content").exists()
WORK = pathlib.Path("/kaggle/working" if IN_KAGGLE else "/content" if IN_COLAB else ".").resolve()
OUT  = WORK / "artifacts"; OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"env: {'kaggle' if IN_KAGGLE else 'colab' if IN_COLAB else 'local'} | device: {DEVICE} | out: {OUT}")

In [ ]:
# ---- dataset location ----
if IN_KAGGLE:
    DATA = pathlib.Path("/kaggle/input/network-intrusion-dataset")
elif IN_COLAB:
    kj = pathlib.Path("/root/.kaggle/kaggle.json")
    if not kj.exists():
        from google.colab import files
        print("upload kaggle.json:")
        up = files.upload()
        kj.parent.mkdir(parents=True, exist_ok=True)
        kj.write_bytes(list(up.values())[0]); kj.chmod(0o600)
    os.system("pip -q install kaggle")
    os.system("kaggle datasets download -d chethuhn/network-intrusion-dataset -p /content/data --unzip")
    DATA = pathlib.Path("/content/data")
else:
    DATA = pathlib.Path("./data")   # local: put the CSVs here

csvs = sorted(p for p in DATA.glob("**/*.csv"))
assert csvs, f"no CSVs under {DATA}"
print(len(csvs), "files:")
for c in csvs: print("  ", c.name)

In [ ]:
# ---- clone the repo, import the shared model definition ----
if not pathlib.Path("HORIZON").exists():
    assert os.system(f"git clone -q {REPO_URL} HORIZON") == 0, "git clone failed"
sys.path.insert(0, str(pathlib.Path("HORIZON/horizon-api").resolve()))

try:
    from horizon_api import FEATURE_KEYS, HISTORY, HORIZON, N_INPUT, N_FEATURES
    from horizon_api.features import FeatureScaler
    from horizon_api.model import HorizonModel, ModelConfig
    from horizon_api.model import save as save_model
    from horizon_api.rollout import rollout, make_intervention
except ModuleNotFoundError as e:
    raise SystemExit("horizon-api/ not found in the cloned repo - push it to GitHub first.") from e

FEAT = list(FEATURE_KEYS)
print("model contract OK. features:", FEAT)
print("HISTORY", HISTORY, "HORIZON", HORIZON, "N_INPUT", N_INPUT)

## Gate 0 - column verification

The state design needs source/dest IP, dest port, and a timestamp. Some cleaned CIC dumps drop them.

In [ ]:
probe = pd.read_csv(csvs[0], nrows=300, low_memory=False)
probe.columns = probe.columns.str.strip()
NEED = ["Source IP", "Destination IP", "Destination Port", "Timestamp", "Label"]
missing = [c for c in NEED if c not in probe.columns]
print("have:", [c for c in NEED if c in probe.columns])
if missing:
    raise SystemExit(f"Gate 0 FAIL: missing {missing} -> branch B/C, see technical.md section 0")
print("Gate 0 PASS - branch A, proceed")

In [ ]:
import ipaddress
from collections import defaultdict

def capture_of(name: str) -> str:
    n = name.lower()
    for day in ("monday", "tuesday", "wednesday", "thursday", "friday"):
        if day in n:
            return f"ids2017-{day}"
    return "ids2017-unknown"

def is_internal(ip: str) -> bool:
    try:
        return ipaddress.ip_address(ip).is_private
    except ValueError:
        return False

cap_files = defaultdict(list)
for c in csvs:
    cap_files[capture_of(c.name)].append(c)
print({k: [p.name for p in v] for k, v in cap_files.items()})

## Host-window aggregation

Raw flows -> one 10-feature row per internal host per 60s window, per `technical.md` 1.2. Same-weekday files are concatenated before windowing so window indices are continuous.

In [ ]:
USE = ["Source IP", "Destination IP", "Destination Port", "Timestamp", "Flow Duration",
       "Total Length of Fwd Packets", "Total Length of Bwd Packets", "RST Flag Count", "Label"]

def _load_capture(paths):
    frames = []
    for p in paths:
        for chunk in pd.read_csv(p, chunksize=250_000, low_memory=False):
            chunk.columns = chunk.columns.str.strip()
            if QUICK:
                chunk = chunk.sample(frac=0.15, random_state=SEED)
            chunk = chunk[[c for c in USE if c in chunk.columns]].copy()
            chunk["ts"] = pd.to_datetime(chunk["Timestamp"], dayfirst=True, errors="coerce")
            chunk = chunk.dropna(subset=["ts", "Source IP", "Destination IP"])
            chunk = chunk[chunk["Source IP"].map(is_internal)]
            if not chunk.empty:
                frames.append(chunk)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

DEMO_HOSTS = {"ids2017-thursday": "192.168.10.15",
              "ids2017-friday": "192.168.10.50",
              "ids2017-monday": "192.168.10.8"}

def aggregate_capture(cap, paths):
    df = _load_capture(paths)
    if df.empty:
        return pd.DataFrame(), None, None, None
    df["dur_s"] = pd.to_numeric(df["Flow Duration"], errors="coerce").fillna(0) / 1e6
    df["b_out"] = pd.to_numeric(df["Total Length of Fwd Packets"], errors="coerce").fillna(0)
    df["b_in"]  = pd.to_numeric(df["Total Length of Bwd Packets"], errors="coerce").fillna(0)
    df["rst"]   = pd.to_numeric(df.get("RST Flag Count", 0), errors="coerce").fillna(0)
    df["ext"]   = ~df["Destination IP"].map(is_internal)
    df["fail"]  = (df["b_in"] == 0) | (df["rst"] > 0)
    df["mal"]   = df["Label"].astype(str).str.upper().str.strip() != "BENIGN"

    t0 = df["ts"].min()
    df["w"] = ((df["ts"] - t0).dt.total_seconds() // WINDOW_SECONDS).astype(int)

    g = df.groupby(["Source IP", "w"], sort=True)
    agg = g.agg(
        n_flows=("ts", "size"),
        n_distinct_dst_ip=("Destination IP", "nunique"),
        n_distinct_dst_port=("Destination Port", "nunique"),
        fail_ratio=("fail", "mean"),
        bytes_out=("b_out", "sum"),
        bytes_in=("b_in", "sum"),
        mean_duration=("dur_s", "mean"),
        external_ratio=("ext", "mean"),
        n_mal=("mal", "sum"),
    ).reset_index().rename(columns={"Source IP": "host"})
    agg["io_ratio"] = agg["bytes_out"] / (agg["bytes_in"] + 1.0)

    # new_peer_rate: running per-host set of destination IPs, in time order
    dsts = (g["Destination IP"].agg(set).reset_index()
            .rename(columns={"Source IP": "host", "Destination IP": "dsts"})
            .sort_values(["host", "w"]))
    seen, npr = {}, {}
    for host, w, s in zip(dsts.host, dsts.w, dsts.dsts):
        known = seen.setdefault(host, set())
        npr[(host, w)] = len(s - known) / max(1, len(s))
        known |= s
    agg["new_peer_rate"] = [npr[(h, w)] for h, w in zip(agg.host, agg.w)]

    # window label = most common malicious class if enough malicious flows, else benign
    mal_rows = df[df["mal"]]
    if len(mal_rows):
        lab = (mal_rows.groupby(["Source IP", "w"])["Label"]
               .agg(lambda x: x.value_counts().index[0]).reset_index()
               .rename(columns={"Source IP": "host", "Label": "label"}))
        agg = agg.merge(lab, on=["host", "w"], how="left")
    else:
        agg["label"] = np.nan
    agg["label"] = agg["label"].where(agg["n_mal"] >= LABEL_MIN_MALICIOUS, "benign").fillna("benign")
    agg["label"] = agg["label"].astype(str).str.strip()
    agg["capture"] = cap

    # --- macro topology: internal host -> internal host adjacency ---
    di = df[~df["ext"]].copy()
    di["dst"] = di["Destination IP"]
    adj = (di.groupby(["Source IP", "dst"]).size().reset_index(name="flows")
           .rename(columns={"Source IP": "src"}))
    ext_out = df[df["ext"]].groupby("Source IP").size().reset_index(name="flows") \
        .rename(columns={"Source IP": "src"})
    ext_out["dst"] = "ext:internet"
    net_edges = pd.concat([adj, ext_out[["src", "dst", "flows"]]], ignore_index=True)

    # --- micro: flow sample for this capture's demo host ---
    flows_json = None
    dh = DEMO_HOSTS.get(cap)
    if dh is not None and (df["Source IP"] == dh).any():
        fs = df[df["Source IP"] == dh].copy()
        # bias the sample toward malicious windows
        weights = np.where(fs["mal"].values, 5.0, 1.0)
        take = min(400, len(fs))
        idx = np.random.default_rng(SEED).choice(len(fs), size=take, replace=False,
                                                 p=weights / weights.sum())
        fs = fs.iloc[np.sort(idx)]
        flows_json = {
            "schema_version": "v4.0", "capture": cap, "host": dh,
            "flows": [{
                "window_idx": int(w),
                "ts": (t0 + pd.Timedelta(seconds=int(w) * WINDOW_SECONDS)).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "dst_ip": str(d), "dst_port": int(pd.to_numeric(p, errors="coerce") or 0),
                "bytes_out": int(bo), "bytes_in": int(bi),
                "label": str(lb).strip(), "internal": not bool(e),
            } for w, d, p, bo, bi, lb, e in zip(
                fs["w"], fs["Destination IP"], fs["Destination Port"],
                fs["b_out"], fs["b_in"], fs["Label"], fs["ext"])],
        }
    return agg, t0, net_edges, flows_json

parts, cap_t0, cap_edges, cap_flows = [], {}, {}, {}
for cap, paths in cap_files.items():
    a, t0, net, flows_json = aggregate_capture(cap, paths)
    cap_t0[cap] = t0
    cap_edges[cap] = net
    if flows_json is not None:
        cap_flows[cap] = flows_json
    print(f"{cap:20s} {len(a):7d} host-windows  ({a.host.nunique() if len(a) else 0} hosts)")
    if len(a):
        parts.append(a)
raw_states = pd.concat(parts, ignore_index=True)

In [ ]:
# ---- fill empty windows, drop short hosts, order columns -> states.parquet ----
def finish(raw):
    out = []
    for (cap, host), grp in raw.groupby(["capture", "host"], sort=False):
        grp = grp.set_index("w").sort_index()
        full = grp.reindex(range(int(grp.index.min()), int(grp.index.max()) + 1))
        full["is_empty"] = full["n_flows"].isna()
        full[FEAT] = full[FEAT].fillna(0.0)
        full["label"] = full["label"].fillna("benign")
        if (~full["is_empty"]).sum() < MIN_WINDOWS:
            continue
        full = full.reset_index().rename(columns={"index": "window_idx"})
        full["capture"], full["host"] = cap, host
        t0 = cap_t0[cap]
        full["ts"] = [(t0 + pd.Timedelta(seconds=int(w) * WINDOW_SECONDS)).strftime("%Y-%m-%dT%H:%M:%SZ")
                      for w in full["window_idx"]]
        out.append(full)
    s = pd.concat(out, ignore_index=True)
    return s[["capture", "host", "window_idx", "ts", *FEAT, "is_empty", "label"]] \
        .sort_values(["capture", "host", "window_idx"]).reset_index(drop=True)

states = finish(raw_states)
states.to_parquet(OUT / "states.parquet", index=False)
print(states.shape, "->", OUT / "states.parquet")
print(states.label.value_counts().head(20))

## Scene data: topology + flow sample

`network_<capture>.json` (macro host-comm graph) and `flows_<capture>_<host>.json` (micro per-host flow sample) for the frontend scene. `states.parquet` does not keep peer identity, so these are built here from the raw flows.

In [ ]:
KIND_HINT = {"1": "gateway", "3": "domain-controller", "5": "server", "16": "server", "19": "server"}

def node_kind(host, in_deg, out_deg):
    last = host.rsplit(".", 1)[-1]
    if host.startswith("ext"):
        return "external"
    if last in KIND_HINT:
        return KIND_HINT[last]
    if in_deg >= 8 and in_deg > out_deg * 2:
        return "server"
    return "workstation"

kept = set(states["host"].unique()) | {"ext:internet"}
for cap in cap_files:
    edges_df = cap_edges.get(cap)
    if edges_df is None or not len(edges_df):
        continue
    e = edges_df[edges_df["src"].isin(kept) & edges_df["dst"].isin(kept)]
    e = e[e["flows"] >= 3].sort_values("flows", ascending=False).head(400)
    indeg = e.groupby("dst")["flows"].sum().to_dict()
    outdeg = e.groupby("src")["flows"].sum().to_dict()
    g = states[states["capture"] == cap].groupby("host").agg(
        n_flows=("n_flows", "sum"), n_windows=("window_idx", "count")).to_dict("index")
    hosts_in = sorted(set(e["src"]) | set(e["dst"]))
    dh = DEMO_HOSTS.get(cap)
    nodes = [{
        "host": h,
        "subnet": "external" if h.startswith("ext") else h.rsplit(".", 1)[0],
        "n_flows": int(g.get(h, {}).get("n_flows", indeg.get(h, 0) + outdeg.get(h, 0))),
        "n_windows": int(g.get(h, {}).get("n_windows", 0)),
        "kind": node_kind(h, indeg.get(h, 0), outdeg.get(h, 0)),
        "is_demo": h == dh,
        "is_target": node_kind(h, indeg.get(h, 0), outdeg.get(h, 0)) == "domain-controller",
    } for h in hosts_in]
    edges = [{"src": r.src, "dst": r.dst, "flows": int(r.flows),
              "internal": not str(r.dst).startswith("ext")} for r in e.itertuples()]
    net = {"schema_version": "v4.0", "capture": cap, "nodes": nodes, "edges": edges}
    (OUT / f"network_{cap}.json").write_text(json.dumps(net, indent=1))
    print(f"network_{cap}.json: {len(nodes)} nodes, {len(edges)} edges")

for cap, fj in cap_flows.items():
    (OUT / f"flows_{cap}_{fj['host']}.json").write_text(json.dumps(fj, indent=1))
    print(f"flows_{cap}_{fj['host']}.json: {len(fj['flows'])} flows")

## Transforms

`FeatureScaler` = log1p on the heavy-tailed subset, then standardise. Fit on the **train split only** (`HELDOUT_CAPTURE` excluded), skipping each host's warm-up windows.

In [ ]:
train_cap = states["capture"] != HELDOUT_CAPTURE
pos_in_host = states.groupby(["capture", "host"]).cumcount()
fit_rows = states[train_cap & (pos_in_host >= WARMUP_WINDOWS) & (~states["is_empty"])]
print("scaler fit rows:", len(fit_rows))

scaler = FeatureScaler().fit(fit_rows[FEAT].to_numpy())
scaler.save(OUT / "scaler.json")

Z_ALL = scaler.transform(states[FEAT].to_numpy()).astype(np.float32)
print("z mean:", Z_ALL.mean(0).round(2))
print("z std :", Z_ALL.std(0).round(2))

## Sequences and splits

One training example per window `i >= 20`: history `z[i-20:i]` (+ zero intervention channel), MDN target `z[i]`, readout target = does an attack window fall in `[i, i+20)`. Grouped by `(capture, host)`; `HELDOUT_CAPTURE` is the out-of-domain test set, plus a 15% host slice of the train captures for early stopping.

In [ ]:
host_index = {}
off = 0
for (cap, host), grp in states.groupby(["capture", "host"], sort=False):
    host_index[(cap, host)] = (off, off + len(grp), grp)
    off += len(grp)

rng = np.random.default_rng(SEED)
train_hosts, val_hosts, test_hosts = [], [], []
for key, (_, _, grp) in host_index.items():
    cap = key[0]
    if cap == HELDOUT_CAPTURE:
        test_hosts.append(key)
    elif rng.random() < 0.15:
        val_hosts.append(key)
    else:
        train_hosts.append(key)

def make_xy(keys, drop_class=None):
    X, Ynext, Yatk = [], [], []
    zero_col = np.zeros((HISTORY, 1), dtype=np.float32)
    for key in keys:
        lo, hi, grp = host_index[key]
        z = Z_ALL[lo:hi]
        lab = grp["label"].to_numpy()
        if drop_class is not None and (lab == drop_class).any():
            continue  # held-out-class: remove hosts that ever show the class
        atk = lab != "benign"
        for i in range(HISTORY, len(grp) - 1):
            X.append(np.concatenate([z[i - HISTORY:i], zero_col], axis=1))
            Ynext.append(z[i])
            Yatk.append(float(atk[i:i + HORIZON].any()))
    return (np.asarray(X, np.float32), np.asarray(Ynext, np.float32), np.asarray(Yatk, np.float32))

Xtr, Ntr, Atr = make_xy(train_hosts)
Xva, Nva, Ava = make_xy(val_hosts)
print("train", Xtr.shape, "| val", Xva.shape, "| attack rate tr/va:", Atr.mean().round(3), Ava.mean().round(3))

## MDN sanity check on toy data

Before real training: can the MDN head separate a deliberately bimodal target? If not it will not work on real data (`implementation.md` Week 1).

In [ ]:
_toy_cfg = ModelConfig()
_toy = HorizonModel(_toy_cfg).to(DEVICE)
_opt = torch.optim.Adam(_toy.parameters(), lr=1e-3)
# next value is +2 with p=0.6, -2 with p=0.4, regardless of history
for step in range(300):
    x = torch.randn(256, HISTORY, N_INPUT, device=DEVICE)
    sign = torch.where(torch.rand(256, device=DEVICE) < 0.6, 2.0, -2.0)
    tgt = (sign[:, None] + 0.1 * torch.randn(256, N_FEATURES, device=DEVICE))
    st, *_ = _toy.encode(x)
    loss = _toy.mdn.nll(st, tgt)
    _opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(_toy.parameters(), 1.0)
    _opt.step()
with torch.no_grad():
    st, *_ = _toy.encode(torch.randn(2000, HISTORY, N_INPUT, device=DEVICE))
    samp = _toy.mdn.sample(st)[:, 0].cpu().numpy()
frac_pos = (samp > 0).mean()
print(f"toy: sampled P(+mode) = {frac_pos:.2f}  (target 0.60)  final nll {loss.item():.3f}")
assert 0.45 < frac_pos < 0.75, "MDN did not separate the bimodal toy target - investigate before real training"
print("MDN sanity OK")

## Train

Joint loss `mdn_nll + LAMBDA_BCE * bce`, both logged separately. Scheduled sampling ramps a short model-fed unroll from 0 to 0.5 over training (`technical.md` 2.5). Checkpoint every epoch to `artifacts/`.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def loader(X, N, A, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(N), torch.from_numpy(A))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=shuffle)

def train_model(Xtr, Ntr, Atr, Xva, Nva, Ava, tag="model", epochs=EPOCHS):
    torch.manual_seed(SEED)
    model = HorizonModel(ModelConfig()).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    pos_w = torch.tensor([(1 - Atr.mean()) / max(Atr.mean(), 1e-3)], device=DEVICE)
    bce = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w)
    tl, vl = loader(Xtr, Ntr, Atr, True), loader(Xva, Nva, Ava, False)

    best, best_state, bad = 1e9, None, 0
    hist = []
    for ep in range(epochs):
        p_ss = 0.5 * ep / max(1, epochs - 1)
        model.train(); m_sum = b_sum = n = 0
        for xb, nb, ab in tl:
            xb, nb, ab = xb.to(DEVICE), nb.to(DEVICE), ab.to(DEVICE)
            state, ctx, _, hidden = model.encode(xb)
            l_mdn = model.mdn.nll(state, nb)
            l_bce = bce(model.readout(state), ab)
            # scheduled-sampling: one extra step fed by the model's own sample
            if p_ss > 0 and torch.rand(1).item() < p_ss:
                s = model.mdn.sample(state).detach()
                x_next = torch.cat([s, torch.zeros(s.size(0), 1, device=DEVICE)], -1)
                state2, _ = model.step(x_next, hidden, ctx)
                l_bce = l_bce + bce(model.readout(state2), ab)
            loss = l_mdn + LAMBDA_BCE * l_bce
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            m_sum += l_mdn.item(); b_sum += l_bce.item(); n += 1

        model.eval(); vm = vb = vn = 0
        with torch.no_grad():
            for xb, nb, ab in vl:
                xb, nb, ab = xb.to(DEVICE), nb.to(DEVICE), ab.to(DEVICE)
                st, *_ = model.encode(xb)
                vm += model.mdn.nll(st, nb).item()
                vb += bce(model.readout(st), ab).item(); vn += 1
        v = vm / vn + vb / vn
        sched.step(v)
        hist.append({"epoch": ep, "mdn": m_sum / n, "bce": b_sum / n, "val": v, "p_ss": round(p_ss, 3)})
        print(f"ep {ep:2d}  mdn {m_sum/n:7.3f}  bce {b_sum/n:6.3f}  val {v:7.3f}  p_ss {p_ss:.2f}")
        save_model(model, OUT / f"{tag}.pt")
        if v < best - 1e-3:
            best, best_state, bad = v, {k: t.cpu().clone() for k, t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 7:
                print("early stop"); break
    if best_state:
        model.load_state_dict(best_state)
        save_model(model, OUT / f"{tag}.pt")
    return model, hist

model, hist = train_model(Xtr, Ntr, Atr, Xva, Nva, Ava)

## Persistence gate

Hard gate (`technical.md` 3.1): the model must beat `s_hat_{t+1} = s_t` on per-feature MSE. If not, it collapsed to the mean.

In [ ]:
@torch.no_grad()
def per_feature_mse(model, X, N):
    model.eval()
    preds, base = [], []
    for i in range(0, len(X), 1024):
        xb = torch.from_numpy(X[i:i + 1024]).to(DEVICE)
        st, *_ = model.encode(xb)
        # mean of the MDN over components as the point prediction
        logits, means, _ = model.mdn(st)
        w = torch.softmax(logits, -1).unsqueeze(-1)
        preds.append((w * means).sum(1).cpu().numpy())
        base.append(xb[:, -1, :N_FEATURES].cpu().numpy())
    pred = np.concatenate(preds); last = np.concatenate(base)
    mdl = ((pred - N) ** 2).mean(0)
    per = ((last - N) ** 2).mean(0)
    return mdl, per

mdl_mse, per_mse = per_feature_mse(model, Xva, Nva)
beat = mdl_mse < per_mse
print(f"{'feature':22s} {'model':>8s} {'persist':>8s}  beat")
for k, m, p, b in zip(FEAT, mdl_mse, per_mse, beat):
    print(f"{k:22s} {m:8.4f} {p:8.4f}  {'yes' if b else 'NO'}")
print(f"\nGATE: model beats persistence on {beat.sum()}/{len(FEAT)} features")
if beat.sum() < len(FEAT) * 0.6:
    print("WARNING: weak - train longer (EPOCHS=40), check scaler leakage, or QUICK=False")

## Rollout check + metrics.json

Run the real 50-sample rollout on a few hosts, sanity-check the curves, write `metrics.json` (persistence table + attack-rate calibration). The full lead-time-vs-FPR curve is P2's Week-3 harness; this is the subset the panel needs.

In [ ]:
demo_hosts = [("ids2017-thursday", "192.168.10.15"),
              ("ids2017-friday", "192.168.10.50"),
              ("ids2017-monday", "192.168.10.8")]

model.eval()
for cap, host in demo_hosts:
    key = (cap, host)
    if key not in host_index:
        print(f"{cap}/{host}: not in states (skipped)"); continue
    lo, hi, grp = host_index[key]
    z = Z_ALL[lo:hi]
    if len(z) < HISTORY + 5:
        print(f"{cap}/{host}: too short"); continue
    t = min(len(z) - 1, HISTORY + (len(z) - HISTORY) // 2)
    res = rollout(model, z[t - HISTORY:t], n_samples=N_ROLLOUT_SAMPLES, seed=0)
    print(f"{cap}/{host} t={t}: p_frac peak {res.p_frac.max():.2f}  divergence {res.divergence:.3f}")

# eval attack-probability calibration on val
@torch.no_grad()
def val_probs(model, X, A):
    out = []
    for i in range(0, len(X), 1024):
        st, *_ = model.encode(torch.from_numpy(X[i:i+1024]).to(DEVICE))
        out.append(model.readout.prob(st).cpu().numpy())
    return np.concatenate(out)

pv = val_probs(model, Xva, Ava)
bins = np.linspace(0, 1, 6)
reliability = []
for a, b in zip(bins[:-1], bins[1:]):
    m = (pv >= a) & (pv < b)
    if m.sum():
        reliability.append({"p_pred": round((a + b) / 2, 2), "p_obs": round(float(Ava[m].mean()), 3)})

metrics = {
    "schema_version": "v4.0",
    "status": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "lead_time_vs_fpr": [],  # filled by P2's harness
    "reconstruction": {
        "persistence_beaten": bool(beat.sum() >= len(FEAT) * 0.6),
        "per_feature_mse": {
            "HORIZON": {k: round(float(v), 4) for k, v in zip(FEAT, mdl_mse)},
            "persistence": {k: round(float(v), 4) for k, v in zip(FEAT, per_mse)},
        },
    },
    "rollout_error_growth": [],
    "generalisation": {"held_out_capture": {}, "held_out_class": {}},
    "standard": {"per_class_f1": {}},
    "calibration": {"platt_slope": None, "reliability": reliability},
    "divergence_auc": None,
    "training_history": hist,
}
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=1))
print("metrics.json written")

## scenarios.json

Ground truth for the demo hosts, read from the real aggregated data. The backend merges this over its built-in table.

In [ ]:
CLASS_HINT = {"ids2017-thursday": "Infiltration", "ids2017-friday": "Bot", "ids2017-monday": "benign"}
HELD_HINT  = {"ids2017-thursday": "PortScan", "ids2017-friday": "Botnet", "ids2017-monday": None}

scenarios = {}
for cap, host in demo_hosts:
    key = (cap, host)
    if key not in host_index:
        continue
    _, _, grp = host_index[key]
    atk_w = grp.loc[grp["label"] != "benign", "window_idx"]
    first = int(atk_w.min()) if len(atk_w) else None
    n = len(grp)
    scenarios[f"{cap}/{host}"] = {
        "scenario": {"ids2017-thursday": "infiltration", "ids2017-friday": "botnet",
                     "ids2017-monday": "benign"}.get(cap, "adhoc"),
        "true_class": CLASS_HINT.get(cap, "unknown"),
        "first_attack_window": first,
        "available_t": sorted({HISTORY + 5, n // 2, max(HISTORY + 5, n - 2)}),
        "held_out": HELD_HINT.get(cap),
    }
(OUT / "scenarios.json").write_text(json.dumps(scenarios, indent=1))
print(json.dumps(scenarios, indent=1))

## Held-out-class experiment (optional)

Set `RUN_HELDOUT_CLASS = True` in config. Retrains from scratch with every host that ever shows `HELDOUT_CLASS` removed, saves `model_heldout_<class>.pt`. The backend uses it for the surprise overlay - does surprise still flag the unseen class?

In [ ]:
if RUN_HELDOUT_CLASS:
    Xh, Nh, Ah = make_xy(train_hosts, drop_class=HELDOUT_CLASS)
    Xhv, Nhv, Ahv = make_xy(val_hosts, drop_class=HELDOUT_CLASS)
    print(f"held-out '{HELDOUT_CLASS}': train {Xh.shape}")
    ho_model, _ = train_model(Xh, Nh, Ah, Xhv, Nhv, Ahv, tag=f"model_heldout_{HELDOUT_CLASS}")
    print(f"saved model_heldout_{HELDOUT_CLASS}.pt")
else:
    print("skipped (RUN_HELDOUT_CLASS = False)")

## Package the artifacts

Download `artifacts/` and drop its contents into `horizon-api/artifacts/`, then restart uvicorn (see `horizon-api/README.md` step 3).

In [ ]:
import shutil
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:32s} {f.stat().st_size/1024:8.1f} KB")

zip_path = shutil.make_archive(str(WORK / "horizon_artifacts"), "zip", OUT)
print("\nzip:", zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
elif IN_KAGGLE:
    print("Kaggle: download from the Output tab (artifacts/ and horizon_artifacts.zip).")
else:
    print("local: artifacts are in", OUT)